# Etapa 2 — Modelagem e Avaliação da MLP

Notebook experimental autocontido para a Etapa 2 do Tech Challenge: construção da MLP em PyTorch, treino com batching e early stopping, comparação com baselines, análise de custo e registro final no MLflow.

Este notebook evita imports de `src/` de propósito: a etapa é exploratória e deve permitir experimentar arquitetura, parâmetros e thresholds antes de refatorar o código de produção.

## 1. Imports, paths, seeds e MLflow

Configura ambiente, fixa seeds e define os diretórios usados para dados, artefatos e tracking local do MLflow.

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from mlflow.models import infer_signature
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "etapa2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
EXPERIMENT_NAME = "telco-churn-mlp-notebooks"
MLFLOW_TRACKING_DIR = NOTEBOOKS_DIR / "mlruns"
MLFLOW_TRACKING_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seeds(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False


set_seeds(RANDOM_SEED)
print(f"project_root={PROJECT_ROOT}")
print(f"device={DEVICE}")
print(f"mlflow_tracking_uri={mlflow.get_tracking_uri()}")

## 2–3. Carga dos XLSX, target, leakage e split

Replica inline a união das tabelas brutas. Depois cria `target`, remove IDs/leakage e faz split estratificado 64/16/20.

In [ ]:
RAW_DATA_FILES = {
    "demographics": "Telco_customer_churn_demographics.xlsx",
    "location": "Telco_customer_churn_location.xlsx",
    "services": "Telco_customer_churn_services.xlsx",
    "population": "Telco_customer_churn_population.xlsx",
    "status": "Telco_customer_churn_status.xlsx",
}
ID_COLUMNS = ("CustomerID", "ID")
LEAKAGE_COLUMNS = (
    "ChurnLabel",
    "ChurnValue",
    "CustomerStatus",
    "ChurnScore",
    "ChurnScoreCategory",
    "ChurnCategory",
    "ChurnReason",
)
TARGET_COLUMN = "target"


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    renamed = {
        column: str(column).strip().replace(" ", "").replace("_", "")
        for column in df.columns
    }
    return df.rename(columns=renamed)


def load_raw_tables(data_dir: Path = DATA_DIR) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    for table_name, file_name in RAW_DATA_FILES.items():
        path = data_dir / file_name
        table = pd.read_excel(path, sheet_name=0)
        tables[table_name] = clean_column_names(table).drop(columns=["Count"], errors="ignore")
    return tables


def load_telco_dataset(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    tables = load_raw_tables(data_dir)
    dataset = (
        tables["demographics"]
        .merge(tables["location"], on="CustomerID")
        .merge(tables["services"], on="CustomerID")
        .merge(tables["population"], on="ZipCode")
        .merge(tables["status"], on="CustomerID")
        .drop(columns=["ID"], errors="ignore")
    )
    if "ChurnValue" not in dataset.columns:
        raise ValueError("Coluna ChurnValue não encontrada para construir o target.")
    dataset[TARGET_COLUMN] = (dataset["ChurnValue"] > 0).astype(int)
    return dataset


df = load_telco_dataset(DATA_DIR)
columns_to_drop = list(ID_COLUMNS) + list(LEAKAGE_COLUMNS) + [TARGET_COLUMN]
X = df.drop(columns=columns_to_drop, errors="ignore")
y = df[TARGET_COLUMN]

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=VAL_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_SEED,
)
x_test_raw = x_test.copy()
y_test_array = y_test.to_numpy()

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(x_train), "churn_rate": y_train.mean()},
        {"split": "validation", "rows": len(x_val), "churn_rate": y_val.mean()},
        {"split": "test", "rows": len(x_test), "churn_rate": y_test.mean()},
    ]
)
print(f"dataset_shape={df.shape}")
print(f"feature_shape={X.shape}")
split_summary.style.format({"churn_rate": "{:.2%}"})

## 4. Pré-processamento e DataLoaders

Imputa e escala variáveis numéricas, imputa e aplica one-hot nas categóricas. O `fit` do pré-processador acontece apenas no treino.

In [ ]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(x: pd.DataFrame) -> ColumnTransformer:
    numeric_features = x.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = x.select_dtypes(exclude=["number", "bool"]).columns.tolist()
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )


def as_float_tensor(values: Any) -> torch.Tensor:
    if hasattr(values, "toarray"):
        values = values.toarray()
    return torch.tensor(values, dtype=torch.float32)


preprocessor = build_preprocessor(x_train)
x_train_processed = preprocessor.fit_transform(x_train)
x_val_processed = preprocessor.transform(x_val)
x_test_processed = preprocessor.transform(x_test)

x_train_tensor = as_float_tensor(x_train_processed)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
x_val_tensor = as_float_tensor(x_val_processed)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1, 1)
x_test_tensor = as_float_tensor(x_test_processed)
INPUT_DIM = x_train_tensor.shape[1]


def make_loaders(batch_size: int) -> tuple[DataLoader, DataLoader]:
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
    )


numeric_features = x_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = x_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()
try:
    feature_names = preprocessor.get_feature_names_out().tolist()
except Exception:
    feature_names = [f"feature_{i}" for i in range(INPUT_DIM)]

feature_summary = pd.DataFrame(
    [
        {"feature_group": "numeric_original", "count": len(numeric_features)},
        {"feature_group": "categorical_original", "count": len(categorical_features)},
        {"feature_group": "processed_input_dim", "count": INPUT_DIM},
    ]
)
feature_summary

## 5. Arquitetura da MLP

A saída é um logit, não uma probabilidade. A sigmoid é aplicada apenas na avaliação, porque `BCEWithLogitsLoss` combina sigmoid + BCE de forma numericamente mais estável.

```mermaid
flowchart LR
    inputLayer[Input features] --> hidden1[Linear inputDim to hiddenDim]
    hidden1 --> relu1[ReLU]
    relu1 --> dropout1[Dropout]
    dropout1 --> hidden2[Linear hiddenDim to halfHidden]
    hidden2 --> relu2[ReLU]
    relu2 --> dropout2[Dropout]
    dropout2 --> outputLayer[Linear halfHidden to 1 logit]
```

In [ ]:
class TelcoMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout_rate: float = 0.2):
        super().__init__()
        half_hidden = max(hidden_dim // 2, 1)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, half_hidden),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(half_hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def parameter_table(model: nn.Module) -> pd.DataFrame:
    rows = []
    total = 0
    for name, parameter in model.named_parameters():
        n_params = int(parameter.numel())
        rows.append({"parameter": name, "shape": tuple(parameter.shape), "n_params": n_params})
        total += n_params
    rows.append({"parameter": "TOTAL", "shape": "", "n_params": total})
    return pd.DataFrame(rows)


example_model = TelcoMLP(INPUT_DIM, hidden_dim=64, dropout_rate=0.2)
parameter_table(example_model)

## 6. Loop de treinamento e early stopping

O loop mantém histórico por época para análise de curva de aprendizado. O early stopping restaura os melhores pesos observados no monitor escolhido.

In [ ]:
def safe_roc_auc(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, probabilities))
    except ValueError:
        return float("nan")


def safe_pr_auc(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    try:
        return float(average_precision_score(y_true, probabilities))
    except ValueError:
        return float("nan")


def evaluate_binary(
    y_true: np.ndarray | pd.Series,
    probabilities: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    y_true_array = np.asarray(y_true)
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true_array, predictions)),
        "precision": float(precision_score(y_true_array, predictions, zero_division=0)),
        "recall": float(recall_score(y_true_array, predictions, zero_division=0)),
        "f1": float(f1_score(y_true_array, predictions, zero_division=0)),
        "roc_auc": safe_roc_auc(y_true_array, probabilities),
        "pr_auc": safe_pr_auc(y_true_array, probabilities),
    }


def finite_metrics(metrics: dict[str, float]) -> dict[str, float]:
    return {key: value for key, value in metrics.items() if math.isfinite(value)}


class EarlyStopping:
    def __init__(
        self,
        patience: int,
        min_delta: float,
        mode: str = "min",
        restore_best_weights: bool = True,
    ) -> None:
        if patience < 1:
            raise ValueError("patience must be >= 1")
        if mode not in {"min", "max"}:
            raise ValueError("mode must be 'min' or 'max'")
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.restore_best_weights = restore_best_weights
        self.best_score: float | None = None
        self.best_epoch: int | None = None
        self.wait_count = 0
        self.early_stopped = False
        self._best_state_dict: dict[str, torch.Tensor] | None = None

    def _is_improvement(self, metric_value: float) -> bool:
        if self.best_score is None:
            return True
        if self.mode == "min":
            return metric_value < self.best_score - self.min_delta
        return metric_value > self.best_score + self.min_delta

    def step(self, metric_value: float, model: nn.Module, epoch: int) -> bool:
        if math.isfinite(metric_value) and self._is_improvement(metric_value):
            self.best_score = metric_value
            self.best_epoch = epoch
            self.wait_count = 0
            if self.restore_best_weights:
                self._best_state_dict = copy.deepcopy(model.state_dict())
            return False
        self.wait_count += 1
        self.early_stopped = self.wait_count >= self.patience
        return self.early_stopped

    def restore(self, model: nn.Module) -> None:
        if self.restore_best_weights and self._best_state_dict is not None:
            model.load_state_dict(self._best_state_dict)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    running_loss = 0.0
    sample_count = 0
    for batch_features, batch_targets in loader:
        batch_features = batch_features.to(device)
        batch_targets = batch_targets.to(device)
        optimizer.zero_grad()
        logits = model(batch_features)
        loss = criterion(logits, batch_targets)
        loss.backward()
        optimizer.step()
        batch_size = batch_features.size(0)
        running_loss += float(loss.item()) * batch_size
        sample_count += batch_size
    return running_loss / sample_count


def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    running_loss = 0.0
    sample_count = 0
    logits_batches: list[torch.Tensor] = []
    target_batches: list[torch.Tensor] = []
    with torch.no_grad():
        for batch_features, batch_targets in loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)
            logits = model(batch_features)
            loss = criterion(logits, batch_targets)
            batch_size = batch_features.size(0)
            running_loss += float(loss.item()) * batch_size
            sample_count += batch_size
            logits_batches.append(logits.detach().cpu())
            target_batches.append(batch_targets.detach().cpu())
    logits = torch.cat(logits_batches).reshape(-1)
    targets = torch.cat(target_batches).reshape(-1).numpy()
    probabilities = torch.sigmoid(logits).numpy()
    metrics = evaluate_binary(targets, probabilities, threshold=0.5)
    return {
        "val_loss": running_loss / sample_count,
        "val_accuracy": metrics["accuracy"],
        "val_precision": metrics["precision"],
        "val_recall": metrics["recall"],
        "val_f1": metrics["f1"],
        "val_roc_auc": metrics["roc_auc"],
        "val_pr_auc": metrics["pr_auc"],
    }


def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    *,
    epochs: int,
    early_stopping: EarlyStopping,
    monitor: str,
    device: torch.device,
    mlflow_logger: Any | None = None,
) -> pd.DataFrame:
    history: list[dict[str, float | int]] = []
    model.to(device)
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = validate_one_epoch(model, val_loader, criterion, device)
        epoch_record = {"epoch": epoch, "train_loss": train_loss, **val_metrics}
        history.append(epoch_record)
        if mlflow_logger is not None:
            mlflow_logger.log_metrics(
                finite_metrics({k: float(v) for k, v in epoch_record.items() if k != "epoch"}),
                step=epoch,
            )
        if early_stopping.step(float(val_metrics[monitor]), model, epoch):
            break
    early_stopping.restore(model)
    return pd.DataFrame(history)


def predict_mlp_probabilities(model: nn.Module, features_tensor: torch.Tensor, device: torch.device = DEVICE) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        logits = model(features_tensor.to(device)).detach().cpu().reshape(-1)
    return torch.sigmoid(logits).numpy()

## 7. Treino canônico da MLP

Configuração inicial para servir como ponto de comparação: `hidden_dim=64`, `dropout=0.2`, `lr=1e-3`, `batch_size=32`, `epochs=100`, `patience=10`.

In [ ]:
comparison_rows: list[dict[str, Any]] = []
model_probabilities: dict[str, np.ndarray] = {}
sklearn_models: dict[str, Pipeline] = {}

CANONICAL_CONFIG = {
    "hidden_dim": 64,
    "dropout_rate": 0.2,
    "learning_rate": 1e-3,
    "batch_size": 32,
    "epochs": 100,
    "patience": 10,
    "min_delta": 1e-4,
    "monitor": "val_loss",
}

set_seeds(RANDOM_SEED)
train_loader, val_loader = make_loaders(CANONICAL_CONFIG["batch_size"])
mlp_canonical = TelcoMLP(
    input_dim=INPUT_DIM,
    hidden_dim=CANONICAL_CONFIG["hidden_dim"],
    dropout_rate=CANONICAL_CONFIG["dropout_rate"],
)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp_canonical.parameters(), lr=CANONICAL_CONFIG["learning_rate"])
early_stopping = EarlyStopping(
    patience=CANONICAL_CONFIG["patience"],
    min_delta=CANONICAL_CONFIG["min_delta"],
    mode="min",
)

with mlflow.start_run(run_name="mlp-canonical") as canonical_run:
    mlflow.log_params({**CANONICAL_CONFIG, "input_dim": INPUT_DIM, "random_seed": RANDOM_SEED, "device": str(DEVICE)})
    history_canonical = fit(
        model=mlp_canonical,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        epochs=CANONICAL_CONFIG["epochs"],
        early_stopping=early_stopping,
        monitor=CANONICAL_CONFIG["monitor"],
        device=DEVICE,
        mlflow_logger=mlflow,
    )
    mlflow.log_metrics({
        "epochs_trained": int(len(history_canonical)),
        "best_epoch": int(early_stopping.best_epoch or 0),
        "best_score": float(early_stopping.best_score or 0.0),
    })
    history_path = OUTPUT_DIR / "mlp_canonical_history.json"
    history_canonical.to_json(history_path, orient="records", indent=2)
    mlflow.log_artifact(str(history_path), artifact_path="training")

canonical_run_id = canonical_run.info.run_id
history_canonical.tail()

## 8. Avaliação do MLP canônico

Métricas finais no test set, matriz de confusão, curvas ROC/PR e curvas de aprendizado.

In [ ]:
def save_learning_curves(history: pd.DataFrame, output_path: Path, title: str) -> Path:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history["epoch"], history["train_loss"], label="train_loss")
    axes[0].plot(history["epoch"], history["val_loss"], label="val_loss")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    for metric in ["val_roc_auc", "val_pr_auc", "val_f1", "val_accuracy"]:
        axes[1].plot(history["epoch"], history[metric], label=metric)
    axes[1].set_title(f"{title}: validation metrics")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    return output_path


def save_confusion_heatmap(y_true: np.ndarray, probabilities: np.ndarray, threshold: float, output_path: Path, title: str) -> Path:
    predictions = (probabilities >= threshold).astype(int)
    matrix = confusion_matrix(y_true, predictions)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
    ax.set_xticklabels(["Stayed", "Churned"])
    ax.set_yticklabels(["Stayed", "Churned"])
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    return output_path


def save_roc_pr_curves(y_true: np.ndarray, probabilities: np.ndarray, output_path: Path, title: str) -> Path:
    fpr, tpr, _ = roc_curve(y_true, probabilities)
    precision, recall, _ = precision_recall_curve(y_true, probabilities)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(fpr, tpr, label=f"ROC-AUC={roc_auc_score(y_true, probabilities):.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_title(f"{title}: ROC")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend()
    axes[1].plot(recall, precision, label=f"PR-AUC={average_precision_score(y_true, probabilities):.3f}")
    axes[1].set_title(f"{title}: Precision-Recall")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    return output_path


mlp_canonical_probabilities = predict_mlp_probabilities(mlp_canonical, x_test_tensor)
metrics_canonical = evaluate_binary(y_test_array, mlp_canonical_probabilities, threshold=0.5)
model_probabilities["MLP-canonical"] = mlp_canonical_probabilities
comparison_rows.append({"model": "MLP-canonical", **metrics_canonical})

learning_curve_path = save_learning_curves(history_canonical, OUTPUT_DIR / "mlp_canonical_learning_curves.png", "MLP canonical")
confusion_path = save_confusion_heatmap(
    y_test_array,
    mlp_canonical_probabilities,
    threshold=0.5,
    output_path=OUTPUT_DIR / "mlp_canonical_confusion_matrix.png",
    title="MLP canonical confusion matrix",
)
roc_pr_path = save_roc_pr_curves(
    y_test_array,
    mlp_canonical_probabilities,
    output_path=OUTPUT_DIR / "mlp_canonical_roc_pr.png",
    title="MLP canonical",
)
canonical_state_path = OUTPUT_DIR / "mlp_canonical_state_dict.pt"
torch.save({"config": CANONICAL_CONFIG, "state_dict": mlp_canonical.state_dict(), "input_dim": INPUT_DIM}, canonical_state_path)

with mlflow.start_run(run_id=canonical_run_id):
    mlflow.log_metrics({f"test_{k}": v for k, v in finite_metrics(metrics_canonical).items()})
    for artifact in [learning_curve_path, confusion_path, roc_pr_path, canonical_state_path]:
        mlflow.log_artifact(str(artifact), artifact_path="evaluation")

pd.DataFrame([metrics_canonical], index=["MLP-canonical"]).style.format("{:.4f}")